## Environment Setup
Do not modify the following two cells. adh_init() details here: [README-adh_init](https://github.com/cfacorp/aao-dbx-helpers/blob/master/README-adh_init.md).

In [0]:
%run /Workspace/Repos/global/aao-dbx-helpers/.boot/adh_init

Looking in indexes: https://[REDACTED]:****@[REDACTED]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.1/98.1 MB 52.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 110.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.9/603.9 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 50.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 146.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 81.8 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-86a20e5b-56d0-411e-b211-3d83bcba750c
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uni

In [0]:
__adh_init()

ADHObject(params=ADHOParamsObject(start_time='1790736658322', parent_run_id='001790736658322', run_id='001790736658322', workspace_directory='/Workspace/Users/su.bae@cfacorp.com/dbx-jobs-cfaadvanalytics-mom-capacity-ml-cc/jobs/mom-capacity-ml-cc-sales-forecast', environment='dev', task_retry_count='0', trigger_date='2026-09-30', task_key='location-attributes', start_date='2026-09-30', global_workspace_directory='/Workspace/Users/su.bae@cfacorp.com/dbx-jobs-cfaadvanalytics-mom-capacity-ml-cc/global_files', trigger_time='1790736658322', task_execution_count='1', workflow_name='mom-capacity-ml-cc-sales-forecast', job_id='0'), conf=ADHConfObject(default_schema='su_bae_test', default_catalog='cfaadvanalyticsdev_primary', default_schema_qualified='cfaadvanalyticsdev_primary.su_bae_test', repo_unique_id='mom-capacity-ml-cc', default_dx_catalog='dx_test', nb_path='/Workspace/Users/su.bae@cfacorp.com/dbx-jobs-cfaadvanalytics-mom-capacity-ml-cc/jobs/mom-capacity-ml-cc-sales-forecast/location-attributes', storage_base='s3://cfaadvanalyticsdev-us-east-1-databricks-primary-data/user_test/su_bae/', redshift_tmpdir='s3://cfaadvanalyticsdev-us-east-1-databricks-primary-data/temp'), is_job=False)

In [0]:
debug = not (adh.conf.default_schema == "mom_capacity_ml_cc" and adh.params.environment in ("dev","qa","prod",))
debug

True

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

from redshift import load_from_redshift
from pyspark.sql.functions import col, when, count
from pyspark.sql import functions as F

## READ Location Data

In [0]:
today = datetime.today()
strategy_open_date = today + relativedelta(years = 3)
strategy_open_date = strategy_open_date.strftime("%Y-%m-%d")

In [0]:

location_query = f"""
SELECT 
    location_number as location_num, 
    location_name, 
    latitude,
    longitude,
    subtype_id AS subtype, 
    CASE 
        WHEN status = 'future' THEN 'pipeline'
        ELSE 'existing' 
    END AS location_type, 
    status AS location_status, 
    state, 
    dma, 
    dma_id,
    submarket, 
    region,
    CASE
        WHEN opened_dt IS NULL THEN
            CASE
                WHEN projected_open_dt IS NULL THEN TO_DATE('{strategy_open_date}', 'YYYY-MM-DD')
                ELSE projected_open_dt
            END
        ELSE opened_dt
    END AS opened_dt,
    hex_string
FROM 
    rd_datamart.conf.location_attributes
WHERE 
    subtype_id IN ('FSU', 'DTO')
    AND status IN ('future', 'open', 'remodel')
    AND country_code = 'US'
"""
location_df = load_from_redshift(location_query)
if debug:
    display(location_df.limit(5))

location_num,location_name,latitude,longitude,subtype,location_type,location_status,state,dma,dma_id,submarket,region,opened_dt,hex_string
01642,The Villages FSU,28.936116,-81.940816,FSU,existing,open,FL,"Orlando-Daytona Beach-Melbourne, FL",534,Orlando-The Villages,Southeast Region,2005-12-01,8844f4ce4dfffff
01692,San Tan Village Marketplace FSU,33.30440543,-111.7434217,FSU,existing,open,AZ,"Phoenix (Prescott), AZ",753,"Phoenix, AZ-North Chandler Gilbert",West Region,2007-09-20,8848eba313fffff
01693,Crossroads Towne Center FSU,33.28190479,-111.7903771,FSU,existing,open,AZ,"Phoenix (Prescott), AZ",753,"Phoenix, AZ-North Chandler Gilbert",West Region,2005-11-17,8848eba207fffff
01849,Camden Town Center FSU,39.102631,-75.543137,FSU,existing,open,DE,"Philadelphia, PA",504,"Dover, DE",Northeast Region,2008-03-27,882aad42d3fffff
01860,McDonough FSU,33.42431,-84.18341,FSU,existing,open,GA,"Atlanta, GA",524,"Atlanta, GA-McDonough-Locust Grove",Southeast Region,2006-02-09,8844c18887fffff


In [0]:
strategy_location_query = f"""
SELECT 
    id AS location_num, 
    location_name, 
    latitude,
    longitude,
    subtype, 
    status AS location_type, 
    strategy_tier AS location_status, 
    state, 
    dma, 
    NULL AS dma_id,
    submarket, 
    region,
    CAST('{strategy_open_date}' AS date) AS opened_dt
FROM 
    conf.integrated_strategy
WHERE 
    subtype IN ('FSU', 'DTO')
    AND status IN ('green', 'yellow')
"""
location_conf_df = load_from_redshift(strategy_location_query)
location_conf_df = location_conf_df.withColumn(f"hex_string", F.expr(f"h3_longlatash3string(longitude, latitude, 8)"))
if debug:
    display(location_conf_df.limit(5))

location_num,location_name,latitude,longitude,subtype,location_type,location_status,state,dma,dma_id,submarket,region,opened_dt,hex_string
4FD71F2B-A42B-4483-B653-27648F1EA4F9,US 15 & Rt 643,38.70927691,-77.78062793,FSU,green,Tier 2,VA,"Washington, DC (Hagerstown, MD)",null,N/A,Northeast Region,2029-09-30,882aaa04dbfffff
036DBB4F-C5D2-473E-A935-6C5817D864A0,111th & Cicero Ave,41.69105634,-87.73981627,FSU,green,Tier 2,IL,"Chicago, IL",null,Chicago-Crestwood,Midwest Region,2029-09-30,8826645729fffff
8355656A-EF5B-44F7-B5B4-8A946D3A2280,Middlesex Turnpike/I-95,42.48421879,-71.21959756,FSU,green,Tier 2,MA,"Boston, MA (Manchester, NH)",null,"Boston, MA | Arlington/Burlington",Northeast Region,2029-09-30,882a306041fffff
44F88519-9854-47F1-8F0F-B49BB164CB2C,Maynardville Pike and Norris Fwy,36.08173891,-83.92733433,FSU,green,Tier 2,TN,"Knoxville, TN",null,Knoxville | Knoxville North,Atlantic Region,2029-09-30,8844ca36b7fffff
D9882866-A3A6-442C-A548-529A55BCA34D,Canal St. & Broad Ave.,29.96784563,-90.08520915,FSU,green,Tier 2,LA,"New Orleans, LA",null,New Orleans - Central,Southwest Region,2029-09-30,884446421bfffff


In [0]:
all_location = location_df.union(location_conf_df)
if debug:
    display(all_location.limit(5))

location_num,location_name,latitude,longitude,subtype,location_type,location_status,state,dma,dma_id,submarket,region,opened_dt,hex_string
00389,Stone Mountain FSU,33.82501324,-84.10540777,FSU,existing,open,GA,"Atlanta, GA",524,"Augusta, GA-Tucker-Stone Mountain",Southeast Region,1996-07-11,8844c10e95fffff
00448,Rivermont Station FSU,33.98656206,-84.27054573,FSU,existing,open,GA,"Atlanta, GA",524,"Atlanta, GA-North Fulton County",Southeast Region,1996-10-10,8844c1231dfffff
00457,Bryan FSU,30.659699,-96.331003,FSU,existing,open,TX,"Waco-Temple-Bryan, TX",625,"Bryan/College Station, TX",Southwest Region,1996-11-07,88446d31a7fffff
00458,Howell Mill FSU,33.80767243,-84.41295334,FSU,existing,open,GA,"Atlanta, GA",524,"Atlanta, GA-Buckhead-Brookhaven-Chamblee",Southeast Region,1997-01-08,8844c1a897fffff
00466,Laurens Road FSU,34.841137,-82.359595,FSU,existing,open,SC,"Greenville-Spartanburg, SC-Asheville, NC-Anderson, SC",567,Greenville - Central,Atlantic Region,1993-03-12,8844dd545bfffff


In [0]:
ssam_loc_query = """
SELECT 
    DISTINCT
    CASE
        WHEN strategy_id != '' then strategy_id
        ELSE location_number
    END AS location_num,
    CASE 
        WHEN is_ssam = 'Yes' THEN 1 
        ELSE 0
    END AS ssam_flag,
    CASE
        WHEN location_name NOT ILIKE '%no dt' THEN 1
        ELSE 0
    END AS no_dt_flag
FROM 
    sbx_gis.mom_location_pipeline_strategy
ORDER BY 
    is_ssam
""" 
ssam_loc_df = load_from_redshift(ssam_loc_query)

ssam_existing = [2869, 4089, 3290, 3256, 2125, 4069, 4738, 1732, 1674, 3140, 1117, 2739, 1758, 3845, 1089, 2465, 2957, 5010, 2143, 2884, 2214, 3434, 2942, 1757, 5090, 4941, 4450, 4769, 3934, 3179, 4269, 3895, 2984, 3742, 1750, 2804, 2981, 3932, 5460, 3418, 2750, 4135, 4585, 2939, 5456, 4584, 4986, 5359, 4633, 5403, 2433, 4473, 4957, 4028, 5369, 4144, 3180, 5167, 3938, 2913, 2940, 4210, 4437, 2248, 4431, 3099, 5299, 5188, 3106, 2877, 4709, 4792, 3898, 2053, 2972, 2788, 4791, 4630, 4985, 5275, 4627, 4761, 2763, 3717, 4140, 3037, 4619, 4114, 4522, 4457, 4498, 5157, 3555, 4390, 5115, 3668, 4421, 3813, 5084, 4193, 3728, 2780, 5373, 5526, 4103, 2583, 5341, 4514, 5134, 4667, 5366, 4695, 3514, 4180, 5235, 4008, 2944, 5423, 2727, 3731, 4253, 4044, 4755, 4743, 5483, 4725, 5411, 4906, 2134, 2674, 1763, 4879, 4955, 5497, 3721, 5458, 4470, 3653, 1881, 5282, 5233, 5156, 5151, 5368, 5234, 5212, 4471, 5093, 5178, 5778, 5792]

ssam_query = """
select 
    location_number,
    site_sale_type
from 
    tririga.design_construction_sites_all
where 
    site_type = 'Site'
    and site_sale_type like '%SSAM%'
"""
samm_loc_df2 = load_from_redshift(ssam_query).toPandas()


ssam_existing = ssam_existing + list(samm_loc_df2.location_number.values)
ssam_existing = [str(x).zfill(5) for x in ssam_existing]


ssam_loc_df = ssam_loc_df.withColumn(
    'ssam_flag',
    when(col('location_num').isin(ssam_existing), 1).otherwise(col('ssam_flag'))
)

if debug:
    display(ssam_loc_df.limit(5))

location_num,ssam_flag,no_dt_flag
01466,0,1
03831,0,1
03870,0,1
05843,0,1
02759,0,1


In [0]:
all_location = all_location.join(ssam_loc_df, on='location_num', how='left')
if debug:
    display(all_location.limit(5))

location_num,location_name,latitude,longitude,subtype,location_type,location_status,state,dma,dma_id,submarket,region,opened_dt,hex_string,ssam_flag,no_dt_flag
00389,Stone Mountain FSU,33.82501324,-84.10540777,FSU,existing,open,GA,"Atlanta, GA",524,"Augusta, GA-Tucker-Stone Mountain",Southeast Region,1996-07-11,8844c10e95fffff,0,1
00448,Rivermont Station FSU,33.98656206,-84.27054573,FSU,existing,open,GA,"Atlanta, GA",524,"Atlanta, GA-North Fulton County",Southeast Region,1996-10-10,8844c1231dfffff,0,1
00457,Bryan FSU,30.659699,-96.331003,FSU,existing,open,TX,"Waco-Temple-Bryan, TX",625,"Bryan/College Station, TX",Southwest Region,1996-11-07,88446d31a7fffff,0,1
00458,Howell Mill FSU,33.80767243,-84.41295334,FSU,existing,open,GA,"Atlanta, GA",524,"Atlanta, GA-Buckhead-Brookhaven-Chamblee",Southeast Region,1997-01-08,8844c1a897fffff,0,1
00466,Laurens Road FSU,34.841137,-82.359595,FSU,existing,open,SC,"Greenville-Spartanburg, SC-Asheville, NC-Anderson, SC",567,Greenville - Central,Atlantic Region,1993-03-12,8844dd545bfffff,0,1


In [0]:
gis_demographics_query = """WITH Mile1 AS (
  SELECT 
    DISTINCT
    location_number AS location_num,
    location_name AS location_name,
    total_population AS total_population_1,
    worker_population AS worker_population_1,
    daytime_population AS daytime_population_1,
    avg_population_growth AS avg_population_growth_1,
    total_households AS total_households_1,
    total_households_per_sq_mile AS total_households_per_sq_mile_1,
    density_classification::int AS density_classification_1,
    med_household_income AS med_household_income_1,
    med_disposable_income AS med_disposable_income_1,
    avg_disposable_income AS avg_disposable_income_1,
    annual_budget_expenditure AS annual_budget_expenditure_1,
    annual_budget_expenditure_index AS annual_budget_expenditure_index_1,
    med_home_value AS med_home_value_1,
    avg_home_value AS avg_home_value_1,
    food_away_from_home AS food_away_from_home_1,
    competitor_count AS competitor_count_1
  FROM conf.primary_market_area
  WHERE distance_from_cfa_location = '1 Miles'
),
Mile3 AS (
  SELECT 
    DISTINCT
    location_number AS location_num,
    location_name AS location_name,
    total_population AS total_population_3,
    worker_population AS worker_population_3,
    daytime_population AS daytime_population_3,
    avg_population_growth AS avg_population_growth_3,
    total_households AS total_households_3,
    total_households_per_sq_mile AS total_households_per_sq_mile_3,
    density_classification::int AS density_classification_3,
    med_household_income AS med_household_income_3,
    med_disposable_income AS med_disposable_income_3,
    avg_disposable_income AS avg_disposable_income_3,
    annual_budget_expenditure AS annual_budget_expenditure_3,
    annual_budget_expenditure_index AS annual_budget_expenditure_index_3,
    med_home_value AS med_home_value_3,
    avg_home_value AS avg_home_value_3,
    food_away_from_home AS food_away_from_home_3,
    competitor_count AS competitor_count_3
  FROM conf.primary_market_area
  WHERE distance_from_cfa_location = '3 Miles'
),
Mile5 AS (
  SELECT 
    DISTINCT
    location_number AS location_num,
    location_name AS location_name,
    total_population AS total_population_5,
    worker_population AS worker_population_5,
    daytime_population AS daytime_population_5,
    avg_population_growth AS avg_population_growth_5,
    total_households AS total_households_5,
    total_households_per_sq_mile AS total_households_per_sq_mile_5,
    density_classification::int AS density_classification_5,
    med_household_income AS med_household_income_5,
    med_disposable_income AS med_disposable_income_5,
    avg_disposable_income AS avg_disposable_income_5,
    annual_budget_expenditure AS annual_budget_expenditure_5,
    annual_budget_expenditure_index AS annual_budget_expenditure_index_5,
    med_home_value AS med_home_value_5,
    avg_home_value AS avg_home_value_5,
    food_away_from_home AS food_away_from_home_5,
    competitor_count AS competitor_count_5
  FROM conf.primary_market_area
  WHERE distance_from_cfa_location = '5 Miles'
)
SELECT 
  DISTINCT
  Mile1.location_num,
  Mile1.total_population_1,
  Mile1.worker_population_1,
  Mile1.daytime_population_1,
  Mile1.avg_population_growth_1,
  Mile1.total_households_1,
  Mile1.total_households_per_sq_mile_1,
  Mile1.density_classification_1,
  Mile1.med_household_income_1,
  Mile1.med_disposable_income_1,
  Mile1.avg_disposable_income_1,
  Mile1.annual_budget_expenditure_1,
  Mile1.annual_budget_expenditure_index_1,
  Mile1.med_home_value_1,
  Mile1.avg_home_value_1,
  Mile1.food_away_from_home_1,
  Mile1.competitor_count_1,
  Mile3.total_population_3,
  Mile3.worker_population_3,
  Mile3.daytime_population_3,
  Mile3.avg_population_growth_3,
  Mile3.total_households_3,
  Mile3.total_households_per_sq_mile_3,
  Mile3.density_classification_3,
  Mile3.med_household_income_3,
  Mile3.med_disposable_income_3,
  Mile3.avg_disposable_income_3,
  Mile3.annual_budget_expenditure_3,
  Mile3.annual_budget_expenditure_index_3,
  Mile3.med_home_value_3,
  Mile3.avg_home_value_3,
  Mile3.food_away_from_home_3,
  Mile3.competitor_count_3,
  Mile5.total_population_5,
  Mile5.worker_population_5,
  Mile5.daytime_population_5,
  Mile5.avg_population_growth_5,
  Mile5.total_households_5,
  Mile5.total_households_per_sq_mile_5,
  Mile5.density_classification_5,
  Mile5.med_household_income_5,
  Mile5.med_disposable_income_5,
  Mile5.avg_disposable_income_5,
  Mile5.annual_budget_expenditure_5,
  Mile5.annual_budget_expenditure_index_5,
  Mile5.med_home_value_5,
  Mile5.avg_home_value_5,
  Mile5.food_away_from_home_5,
  Mile5.competitor_count_5
FROM Mile1
JOIN Mile3 ON Mile3.location_num = Mile1.location_num
JOIN Mile5 ON Mile5.location_num = Mile1.location_num
"""
gis_demographics = load_from_redshift(gis_demographics_query)
all_location_with_demo = all_location.join(gis_demographics, on='location_num', how='left')
all_location_with_demo = all_location_with_demo.filter(col("location_num").isNotNull())
if debug:
  display(all_location_with_demo.limit(5))

location_num,location_name,latitude,longitude,subtype,location_type,location_status,state,dma,dma_id,submarket,region,opened_dt,hex_string,ssam_flag,no_dt_flag,total_population_1,worker_population_1,daytime_population_1,avg_population_growth_1,total_households_1,total_households_per_sq_mile_1,density_classification_1,med_household_income_1,med_disposable_income_1,avg_disposable_income_1,annual_budget_expenditure_1,annual_budget_expenditure_index_1,med_home_value_1,avg_home_value_1,food_away_from_home_1,competitor_count_1,total_population_3,worker_population_3,daytime_population_3,avg_population_growth_3,total_households_3,total_households_per_sq_mile_3,density_classification_3,med_household_income_3,med_disposable_income_3,avg_disposable_income_3,annual_budget_expenditure_3,annual_budget_expenditure_index_3,med_home_value_3,avg_home_value_3,food_away_from_home_3,competitor_count_3,total_population_5,worker_population_5,daytime_population_5,avg_population_growth_5,total_households_5,total_households_per_sq_mile_5,density_classification_5,med_household_income_5,med_disposable_income_5,avg_disposable_income_5,annual_budget_expenditure_5,annual_budget_expenditure_index_5,med_home_value_5,avg_home_value_5,food_away_from_home_5,competitor_count_5
00389,Stone Mountain FSU,33.82501324,-84.10540777,FSU,existing,open,GA,"Atlanta, GA",524,"Augusta, GA-Tucker-Stone Mountain",Southeast Region,1996-07-11,8844c10e95fffff,0,1,6142,4386,7955,0.66,2251,79.61283932,7,83758.00,67841.00,80734.00,207353587.00,92.00,395796.00,415058.00,8518447.00,15,50502,13876,44703,-0.20,17466,617.73338584,5,103614.00,83105.00,91668.00,1837408531.00,105.00,397025.00,433569.00,75327946.00,34,166022,33308,130293,0.02,56413,1995.20173453,4,89262.00,73321.00,82872.00,5378128119.00,95.00,375962.00,417399.00,222477261.00,61
00448,Rivermont Station FSU,33.98656206,-84.27054573,FSU,existing,open,GA,"Atlanta, GA",524,"Atlanta, GA-North Fulton County",Southeast Region,1996-10-10,8844c1231dfffff,0,1,7807,2319,7171,1.21,3425,121.13459559,6,121031.00,96628.00,120155.00,501023643.00,146.00,529820.00,591823.00,21165169.00,5,74367,20346,66150,-0.08,29607,1047.13342233,4,127174.00,102350.00,127138.00,4606464763.00,156.00,587576.00,637092.00,191044503.00,26,215955,112177,245801,-0.08,85715,3031.54798850,3,112462.00,90779.00,115324.00,12136156206.00,142.00,567547.00,613216.00,514409464.00,130
00457,Bryan FSU,30.659699,-96.331003,FSU,existing,open,TX,"Waco-Temple-Bryan, TX",625,"Bryan/College Station, TX",Southwest Region,1996-11-07,88446d31a7fffff,0,1,7682,9400,12684,0.53,3608,127.60689660,6,59860.00,51178.00,69683.00,277837777.00,77.00,242878.00,259085.00,11586843.00,23,76194,44101,83018,0.99,31967,1130.60134805,4,49322.00,43528.00,59371.00,2097598922.00,66.00,284484.00,344117.00,89668435.00,78,158580,86081,184386,0.92,60637,2144.59517446,3,47011.00,41119.00,56229.00,3765349551.00,62.00,263750.00,324243.00,162141981.00,116
00458,Howell Mill FSU,33.80767243,-84.41295334,FSU,existing,open,GA,"Atlanta, GA",524,"Atlanta, GA-Buckhead-Brookhaven-Chamblee",Southeast Region,1997-01-08,8844c1a897fffff,0,1,16774,9583,18784,0.34,9179,324.64071617,6,112308.00,87327.00,115991.00,1372341949.00,150.00,673870.00,715108.00,61430076.00,17,148382,157826,256476,1.72,76521,2706.37675585,3,114281.00,91521.00,122052.00,12228130684.00,160.00,650960.00,791919.00,544675906.00,111,369454,494121,660380,0.94,186443,6594.07223496,2,103216.00,81820.00,110277.00,26538437722.00,142.00,636038.00,763871.00,1170661554.00,271
00466,Laurens Road FSU,34.841137,-82.359595,FSU,existing,open,SC,"Greenville-Spartanburg, SC-Asheville, NC-Anderson, SC",567,Greenville - Central,Atlantic Region,1993-03-12,8844dd545bfffff,0,1,5551,9021,12706,0.92,2619,92.62817689,7,53542.00,44210.00,59848.00,182913605.00,70.00,350240.00,400476.00,7498239.00,12,71872,85430,134507,1.35,34512,1220.61231032,4,80579.00,65634.00,96246.00,4140023753.00,120.00,482901.00,577740.00,176060290.00,80,180238,140433,256807,1.46,80726,2855.09820824,3,71873.00,5

In [0]:
if debug:
    null_counts = all_location_with_demo.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c) for c in all_location_with_demo.columns
    ])
    display(null_counts)

location_num,location_name,latitude,longitude,subtype,location_type,location_status,state,dma,dma_id,submarket,region,opened_dt,hex_string,ssam_flag,no_dt_flag,total_population_1,worker_population_1,daytime_population_1,avg_population_growth_1,total_households_1,total_households_per_sq_mile_1,density_classification_1,med_household_income_1,med_disposable_income_1,avg_disposable_income_1,annual_budget_expenditure_1,annual_budget_expenditure_index_1,med_home_value_1,avg_home_value_1,food_away_from_home_1,competitor_count_1,total_population_3,worker_population_3,daytime_population_3,avg_population_growth_3,total_households_3,total_households_per_sq_mile_3,density_classification_3,med_household_income_3,med_disposable_income_3,avg_disposable_income_3,annual_budget_expenditure_3,annual_budget_expenditure_index_3,med_home_value_3,avg_home_value_3,food_away_from_home_3,competitor_count_3,total_population_5,worker_population_5,daytime_population_5,avg_population_growth_5,total_households_5,total_households_per_sq_mile_5,density_classification_5,med_household_income_5,med_disposable_income_5,avg_disposable_income_5,annual_budget_expenditure_5,annual_budget_expenditure_index_5,med_home_value_5,avg_home_value_5,food_away_from_home_5,competitor_count_5
0,3,18,18,0,0,7,0,21,3809,517,113,0,18,479,479,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68


In [0]:
fill_values = {
    'dma': 'N/A', 
    'dma_id': 'N/A', 
    'submarket': 'N/A',
    'ssam_flag': 0,
    'no_dt_flag': 1
}

all_location_with_demo = all_location_with_demo.fillna(fill_values)

In [0]:
all_location_with_demo.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(f'{adh.conf.default_schema_qualified}.location_features')

print(f'{adh.conf.default_schema_qualified}.location_features')
print(f'row count : {all_location_with_demo.count()}')
if debug:
    display(all_location_with_demo)

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-6109486647001524>, line 5
      1 all_location_with_demo.write.format("delta") \
      2     .mode("overwrite") \
      3     .option("overwriteSchema", True) \
      4     .option("delta.columnMapping.mode", "name") \
----> 5     .saveAsTable(f'{adh.conf.default_schema_qualified}.location_features')
      7 print(f'{adh.conf.default_schema_qualified}.location_features')
      8 print(f'row count : {all_location_with_demo.count()}')

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 s